# Rosati α-β — PreprocessingBuild one clean cloud per `(donor, chain)` from Rosati's MiXCR clonotypes. Same filters as CD4/CD8 plusa **locus filter** (keep only TRA in alpha, TRB in beta — MiXCR sometimes emits TRD/TRG in the TRA exportbecause the TRA/TRD locus overlaps).**Subset:** Study 1, AlphaBeta, v2 version — 209 donors (CD 88, Healthy 85, UC 36).**UMI threshold: parameterized.** Set `MIN_UMI = 1` or `2` and re-run — both versions are produced andcompared downstream. Rosati keeps molecular counts (UMI), so abundance is real; ~90% of clonotypes aresingletons (UMI=1), which is why the two thresholds give very different cloud sizes.

## 1 · Config, manifest, cleaning functions`MIN_UMI` selects the version. The manifest maps each ERR accession → (donor, group).

In [ ]:
import os, re, globimport numpy as npimport pandas as pdMIN_UMI  = 1      # <--- set to 1 or 2 and re-runMIN_CDR3 = 8ROSATI   = '../raw/rosati_mixcr'                       # MiXCR clonotype tables (ERR*.clones_TR{A,B}.tsv)MANIFEST = '../data/rosati/manifest.tsv'               # columns: err, donor, groupOUTDIR   = f'../data/rosati/clouds_raw_umi{MIN_UMI}'os.makedirs(OUTDIR, exist_ok=True)man = pd.read_csv(MANIFEST, sep='\t')print(f'MIN_UMI = {MIN_UMI}  ->  {OUTDIR}')print(f'manifest: {len(man)} donors | {man["group"].value_counts().to_dict()}')

In [ ]:
def clean_vgene(v):    """'TRAV6*00(394.3)' -> 'TRAV6'. None if unparseable."""    if not isinstance(v, str) or not v:        return None    return re.split(r'[\*\(]', v)[0]def preprocess_one(path, expected_locus):    """expected_locus: 'TRA' or 'TRB'. Returns a clean clonotype DataFrame (or None if empty)."""    df = pd.read_csv(path, sep='\t').rename(columns={        'uniqueMoleculeCount': 'umi', 'readCount': 'reads',        'aaSeqCDR3': 'cdr3aa', 'allVHitsWithScore': 'v_raw'})    # 1) productive: starts with C, len>=8, no stop(*) or frameshift(_)    df = df[df['cdr3aa'].notna()]    df = df[df['cdr3aa'].str.startswith('C')]    df = df[df['cdr3aa'].str.len() >= MIN_CDR3]    df = df[~df['cdr3aa'].str.contains(r'[\*_]', regex=True)]    # 2) clean V gene    df['v_gene'] = df['v_raw'].map(clean_vgene)    df = df[df['v_gene'].notna()]    # 3) LOCUS filter: keep only the expected chain (drops TRD/TRG leakage)    df = df[df['v_gene'].str.startswith(expected_locus)]    # 4) UMI threshold    df = df[df['umi'] >= MIN_UMI]    # 5) dedup by (cdr3aa, v_gene), summing UMIs    g = df.groupby(['cdr3aa', 'v_gene'], as_index=False)['umi'].sum()    if len(g) == 0:        return None    # 6) abundance weights w_log = log(1+umi), normalized    g['count'] = g['umi']    g['w_log'] = np.log1p(g['umi']); g['w_log'] = g['w_log'] / g['w_log'].sum()    return g[['cdr3aa', 'v_gene', 'count', 'w_log']]print('functions ready')

## 2 · Filter funnel on one donorHow many clonotypes survive each filter (CD_100 TRA). **Watch step 7 (UMI≥2):** on this RNA data mostclonotypes are singletons, so a UMI≥2 threshold removes ~90% of them.

In [ ]:
test_err = man.iloc[0]['err']; test_donor = man.iloc[0]['donor']path = f'{ROSATI}/{test_err}.clones_TRA.tsv'df = pd.read_csv(path, sep='\t').rename(columns={    'uniqueMoleculeCount':'umi','readCount':'reads','aaSeqCDR3':'cdr3aa','allVHitsWithScore':'v_raw'})print(f'donor {test_donor} | ERR {test_err}')print(f'0. raw ................... {len(df)}')d = df[df['cdr3aa'].notna()];                                     print(f'1. cdr3 not null ......... {len(d)}')d = d[d['cdr3aa'].str.startswith('C')];                          print(f'2. starts with C ......... {len(d)}')d = d[d['cdr3aa'].str.len() >= 8];                               print(f'3. length >= 8 ........... {len(d)}')d = d[~d['cdr3aa'].str.contains(r'[\*_]', regex=True)];          print(f'4. no stop/frameshift .... {len(d)}')d['v_gene'] = d['v_raw'].map(clean_vgene); d = d[d['v_gene'].notna()]; print(f'5. V parseable ........... {len(d)}')d = d[d['v_gene'].str.startswith('TRA')];                        print(f'6. locus TRA ............. {len(d)}')d2 = d[d['umi'] >= 2];                                           print(f'7. UMI >= 2 .............. {len(d2)}   <-- big drop (singletons)')print(f'\nclonotypes with UMI==1: {(d["umi"]==1).sum()} ({(d["umi"]==1).mean()*100:.1f}%)')print(f'total UMI mass: {d["umi"].sum():.0f}')

## 3 · Verify the locus filter on one donorConfirm no TRD/TRG leaks into either chain after filtering.

In [ ]:
for chain, locus in [('A','TRA'), ('B','TRB')]:    path = f'{ROSATI}/{test_err}.clones_{locus}.tsv'    raw = pd.read_csv(path, sep='\t')    cloud = preprocess_one(path, locus)    vraw = raw['allVHitsWithScore'].map(clean_vgene).dropna()    other = vraw[~vraw.str.startswith(locus)].str[:3].value_counts().to_dict()    print(f'{("alpha" if chain=="A" else "beta"):5}: clean {len(cloud)} clonotypes | non-{locus} loci in raw: {other}')

## 4 · Run all 209 donors × 2 chainsWrite one parquet cloud per `(donor, chain)`. Skip clouds with <10 clonotypes.

In [ ]:
rows, skipped = [], []for _, r in man.iterrows():    err, donor, group = r['err'], r['donor'], r['group']    for chain, locus in [('A','TRA'), ('B','TRB')]:        path = f'{ROSATI}/{err}.clones_{locus}.tsv'        if not os.path.exists(path):            skipped.append((donor, chain, 'no file')); continue        cloud = preprocess_one(path, locus)        if cloud is None or len(cloud) < 10:            skipped.append((donor, chain, 'too few')); continue        cloud.to_parquet(f'{OUTDIR}/{donor}_{chain}.parquet', index=False)        rows.append({'donor': donor, 'group': group, 'chain': locus, 'n_clonotypes': len(cloud)})summary = pd.DataFrame(rows)print(f'clouds written: {len(summary)} | skipped: {len(skipped)}')if skipped: print('skipped:', skipped[:10])

## 5 · Summary — complete pairs and α vs β depth**Key checks:** how many donors keep BOTH chains (pairs for the experiment), and whether α and β havevery different depths (a possible confound, as in CD4/CD8).

In [ ]:
print('=== clouds by group and chain ===')print(summary.groupby(['group','chain']).size(), '\n')wide = summary.pivot_table(index='donor', columns='chain', values='n_clonotypes')pairs = wide.dropna()print(f'complete pairs (α and β): {len(pairs)} of {summary["donor"].nunique()} donors\n')print('=== depth (n_clonotypes) by chain ===')print(summary.groupby('chain')['n_clonotypes'].describe()[['count','min','25%','50%','75%','max']])wide['ratio_A_B'] = wide['TRA'] / wide['TRB']print(f'\nmedian TRA/TRB depth ratio: {wide["ratio_A_B"].median():.2f}')

## 6 · Cloud-size distribution: UMI≥1 vs UMI≥2Compare the two thresholds directly (reads both folders). Shows how much the UMI≥2 filter shrinks theclouds — the trade-off between cleaner counts and cloud depth.

In [ ]:
import matplotlib.pyplot as pltbase = '../data/rosati'def sizes(umi):    fs = glob.glob(f'{base}/clouds_raw_umi{umi}/*.parquet')    return np.array([len(pd.read_parquet(f, columns=['cdr3aa'])) for f in fs])fig, ax = plt.subplots(figsize=(9, 5.5))for umi, col in [(1, '#2C7FB8'), (2, '#C0392B')]:    try:        s = sizes(umi)        ax.hist(s, bins=40, alpha=0.55, color=col, edgecolor='white', label=f'UMI\u2265{umi} (median {np.median(s):.0f})')    except Exception:        passax.set_xlabel('Clonotypes per cloud', fontsize=12, fontweight='bold')ax.set_ylabel('Number of clouds', fontsize=12, fontweight='bold')ax.set_title('Rosati cloud size distribution: UMI\u22651 vs UMI\u22652', fontsize=12)ax.legend(fontsize=10); ax.grid(True, alpha=0.25)plt.tight_layout(); plt.show()